# 第 2 章：把文本变成模型能使用的数字

**先动手观察形状，再研究公式。** 这一章依次走过「正则分词 → 词表 → GPT-2 BPE → 滑动窗口 → 两种 embedding」。
本章教学函数写在 `chapter02/data.py`。把 `USE_VERDICT` 设成 `True` 就会下载书中的 `the-verdict.txt`。


In [ ]:
from pathlib import Path
import sys

# notebook 在项目根目录或章节目录中打开时，都能找到我们的 Python 文件。
ROOT = Path.cwd()
if not (ROOT / "chapter02").exists():
    ROOT = ROOT.parent
if not (ROOT / "chapter02").exists():
    raise FileNotFoundError("请从本项目的根目录或章节目录启动 Jupyter")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("项目目录:", ROOT)


## 1. 用正则表达式拆开文字

正则中的 `|` 表示“或者”；括号保留标点、`--` 和特殊 token。空格用于切分，但简易词表里不保留它。
这意味着下面的 `decode` 只能大致还原文本，无法恢复原有空格。


In [ ]:
from chapter02.data import (
    SAMPLE_TEXT, load_verdict, simple_tokenize, build_vocab,
    SimpleTokenizerV2, create_dataloader_v1
)

USE_VERDICT = False  # 首次练习可改为 True，下载 PDF 使用的完整短篇小说
raw_text = load_verdict() if USE_VERDICT else SAMPLE_TEXT
print("字符数:", len(raw_text))
print("前几个 token:", simple_tokenize(raw_text)[:30])


## 2. 给 token 编号，并试试反向查询

`vocab` 是「token → id」字典；反向字典是「id → token」。
`<|endoftext|>` 常作为文本边界；`<|unk|>` 代表简易词表没见过的 token。
简易分词器的局限是：未知词都变成同一个 `<|unk|>`，因此信息会丢失。


In [ ]:
vocab = build_vocab(raw_text)
tokenizer_simple = SimpleTokenizerV2(vocab)
ids = tokenizer_simple.encode("The fox runs. <|endoftext|> A dragon!")
print("词表大小:", len(vocab))
print("特殊 token 的 id:", {s: vocab[s] for s in ("<|endoftext|>", "<|unk|>")})
print("编码:", ids)
print("id 查词:", [tokenizer_simple.int_to_str[i] for i in ids])
print("解码:", tokenizer_simple.decode(ids))


## 3. 换成真正的 GPT-2 分词器

`tiktoken` 使用 BPE。它可以把中文或新词拆成更小的 token；一个 token 不一定等于一个汉字或一个英文单词。
`gpt2` 是编码方案名；`gpt-4o` 是**模型名**，可以用 `encoding_for_model` 查到对应方案。不要写成 `gpt4-o`。
特殊 token 需明确加入 `allowed_special` 才能按一个 token 编码。


In [ ]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")
chinese_ids = tokenizer.encode("狐狸跑过森林。")
print("GPT-2 编码方案:", tokenizer.name)
print("中文 token id:", chinese_ids)
print("解码:", tokenizer.decode(chinese_ids))
print("gpt-4o 对应方案:", tiktoken.encoding_for_model("gpt-4o").name)
print("可用方案:", tiktoken.list_encoding_names())
print("边界 token:", tokenizer.encode("<|endoftext|>", allowed_special={"<|endoftext|>"}))


## 4. 滑动窗口制造训练题目

假设输入 `[a,b,c,d]`，目标就是 `[b,c,d,e]`。每个位置学会猜**下一个** token。
`max_length=8` 是**每个样本**的长度；`stride=4` 是窗口每次向右走 4 格。
这一步为将来训练整个 GPT 做准备，embedding 也会在第 5 章一起更新。


In [ ]:
loader = create_dataloader_v1(
    raw_text, batch_size=2, max_length=8, stride=4,
    shuffle=False, drop_last=False, tokenizer=tokenizer
)
batch_x, batch_y = next(iter(loader))
print("输入形状 [B,T]:", batch_x.shape)
print("目标形状 [B,T]:", batch_y.shape)
print("第一条输入:", batch_x[0].tolist())
print("第一条目标:", batch_y[0].tolist())
assert batch_x[0, 1:].tolist() == batch_y[0, :-1].tolist()


## 5. token embedding + 可学习位置 embedding

`nn.Embedding` 像一张可训练的查询表。词表大小 `V`、向量宽度 `D`，token 表有 `V×D` 个数。
位置表长 `L`，有 `L×D` 个数。`P[0]` 创建时是随机数，训练中会变化；训练结束后，通常同一个位置查到相同的向量。

词向量是 `[B,T,D]`，位置向量是 `[T,D]`。相加时位置向量会沿 batch 方向“复制”给每句话，
最后一维 `D` 必须相同。PDF 此处使用**可学习的位置表**，不是正弦/余弦公式。


In [ ]:
import torch
from torch import nn

D = 8
context_length = 16  # 位置表能查到 0～15；当前样本 T=8，满足 T<=16
token_embedding_layer = nn.Embedding(tokenizer.n_vocab, D)
position_embedding_layer = nn.Embedding(context_length, D)
token_vectors = token_embedding_layer(batch_x)  # [B,T,D]
position_vectors = position_embedding_layer(torch.arange(batch_x.shape[1]))  # [T,D]
input_vectors = token_vectors + position_vectors
print("token 向量:", token_vectors.shape)
print("位置向量:", position_vectors.shape)
print("相加之后:", input_vectors.shape)
print("位置 0 的向量 P[0]:", position_embedding_layer.weight[0].detach())
